# SDRF Pipeline v2 — Per-File Metadata + Full Ontology Normalization

**Architecture:**
- Layer 0: Training SDRF overlap (ground truth if available)
- Layer 1: PRIDE API → organism, tissue, disease, instrument per PXD
- Layer 2: Regex from paper text → protocol parameters (same across PXD)
- Layer 3: **Filename token parser** → per-file replicate, fraction, condition, label
- Layer 4: **Controlled vocabulary normalization** → every string mapped to exact ontology
- Layer 5: Global majority fallback

Key insight: filenames are compressed SDRF rows. `ad_pl01.raw` encodes
Alzheimer, plasma, replicate 1. Parse this and you get per-file metadata
that paper-level extraction cannot produce.

In [1]:
import os, re, json, time, difflib
from collections import defaultdict, Counter
from pathlib import Path

import requests
import pandas as pd

PRIDE_TIMEOUT = 15
PX_TIMEOUT    = 15

IS_KAGGLE   = Path('/kaggle').exists()
BASE_PATH   = Path('/kaggle/input/harmonizing-the-data-of-your-data') if IS_KAGGLE \
              else Path.cwd().parent / 'data'

TRAIN_SDRF_DIR = BASE_PATH / 'Training_SDRFs' / 'HarmonizedFiles'
SAMPLE_SUB     = BASE_PATH / 'SampleSubmission.csv'
OUT_PATH       = Path('/kaggle/working/submission_v2.csv') if IS_KAGGLE \
                 else Path.cwd().parent / 'outputs' / 'submission_v2_per_file.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

_pubtext_candidates = [
    BASE_PATH / 'Test_PubText' / 'Test PubText',
    BASE_PATH / 'Test_PubText',
    BASE_PATH / 'TestPubText',
    BASE_PATH / 'Test PubText',
    BASE_PATH.parent / 'data' / 'TestPubText',
]
TEST_TEXT_DIR = next((p for p in _pubtext_candidates if p.exists()), _pubtext_candidates[0])

print(f'Base path  : {BASE_PATH}')
print(f'PubText    : {TEST_TEXT_DIR} — exists: {TEST_TEXT_DIR.exists()}')
print(f'Output     : {OUT_PATH}')

Base path  : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data
PubText    : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText — exists: True
Output     : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_v2_per_file.csv


## 1. Full Ontology Dictionaries

Every extraction gets normalized through these before writing to the submission.
This is what separates a 0.15 score from a 0.30+ score — the same extracted value
`blood plasma` needs to become `NT=blood plasma;AC=UBERON:0001969` to score correctly.

In [2]:
# ── PSI-MS instrument ontology (MS: accessions) ───────────────────────────
# Key: any substring that could appear in paper text or PRIDE API
# Value: canonical SDRF string NT=...;AC=MS:...
INSTRUMENT_ONT = {
    # Thermo Q Exactive family
    'q exactive hf-x':        'AC=MS:1003027;NT=Q Exactive HF-X',
    'q exactive hf':          'AC=MS:1002523;NT=Q Exactive HF',
    'q exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q-exactive plus':        'AC=MS:1002634;NT=Q Exactive Plus',
    'q exactive':             'AC=MS:1001911;NT=Q Exactive',
    'qexactive':              'AC=MS:1001911;NT=Q Exactive',
    # Thermo Orbitrap family
    'orbitrap astral':        'AC=MS:1003378;NT=Orbitrap Astral',
    'orbitrap fusion lumos':  'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'fusion lumos':           'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'orbitrap fusion':        'AC=MS:1002416;NT=Orbitrap Fusion',
    'orbitrap eclipse':       'AC=MS:1003029;NT=Orbitrap Eclipse',
    'orbitrap exploris 480':  'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'exploris 480':           'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'ltq orbitrap velos':     'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'ltq orbitrap elite':     'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'ltq orbitrap xl':        'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'ltq orbitrap':           'AC=MS:1000449;NT=LTQ Orbitrap',
    'velos pro':              'AC=MS:1001909;NT=LTQ Velos Pro',
    # Bruker timsTOF
    'timstof pro 2':          'AC=MS:1003474;NT=timsTOF Pro 2',
    'timstof pro':            'AC=MS:1003231;NT=timsTOF Pro',
    'timstof':                'AC=MS:1002817;NT=timsTOF',
    # Bruker impact
    'impact ii':              'AC=MS:1002817;NT=impact II',
    'maxi speed':             'AC=MS:1002817;NT=maXis Speed',
    # AB Sciex
    'triple tof 6600':        'AC=MS:1000931;NT=TripleTOF 6600',
    'triple tof 5600':        'AC=MS:1000931;NT=TripleTOF 5600',
    'triple tof':             'AC=MS:1000931;NT=TripleTOF 6600',
    'sciex 6600':             'AC=MS:1000931;NT=TripleTOF 6600',
    # Waters
    'synapt g2-si':           'AC=MS:1002726;NT=Synapt G2-Si',
    'synapt g2':              'AC=MS:1002726;NT=Synapt G2-Si',
    'synapt':                 'AC=MS:1001755;NT=SYNAPT',
}

# ── UBERON tissue ontology ─────────────────────────────────────────────────
TISSUE_ONT = {
    # Brain regions
    'prefrontal cortex':              'NT=prefrontal cortex;AC=UBERON:0000451',
    'frontal cortex':                 'NT=frontal cortex;AC=UBERON:0001870',
    'temporal cortex':                'NT=temporal lobe;AC=UBERON:0001871',
    'cerebral cortex':                'NT=cerebral cortex;AC=UBERON:0000956',
    'cortex':                         'NT=cerebral cortex;AC=UBERON:0000956',
    'hippocampus':                    'NT=hippocampal formation;AC=UBERON:0002421',
    'cerebellum':                     'NT=cerebellum;AC=UBERON:0002037',
    'striatum':                       'NT=striatum;AC=UBERON:0002435',
    'substantia nigra':               'NT=substantia nigra;AC=UBERON:0002038',
    'thalamus':                       'NT=thalamus;AC=UBERON:0001897',
    'brain':                          'NT=brain;AC=UBERON:0000955',
    # Blood and biofluids
    'blood plasma':                   'NT=blood plasma;AC=UBERON:0001969',
    'plasma':                         'NT=blood plasma;AC=UBERON:0001969',
    'blood serum':                    'NT=blood serum;AC=UBERON:0001977',
    'serum':                          'NT=blood serum;AC=UBERON:0001977',
    'whole blood':                    'NT=blood;AC=UBERON:0000178',
    'blood':                          'NT=blood;AC=UBERON:0000178',
    'peripheral blood':               'NT=blood;AC=UBERON:0000178',
    'urine':                          'NT=urine;AC=UBERON:0001088',
    'cerebrospinal fluid':            'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'csf':                            'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'saliva':                         'NT=saliva;AC=UBERON:0001836',
    'bronchoalveolar lavage':         'NT=bronchoalveolar lavage fluid;AC=UBERON:0000177',
    'bal':                            'NT=bronchoalveolar lavage fluid;AC=UBERON:0000177',
    'synovial fluid':                 'NT=synovial fluid;AC=UBERON:0001090',
    'tear':                           'NT=lacrimal fluid;AC=UBERON:0001827',
    # Organs
    'liver':                          'NT=liver;AC=UBERON:0002107',
    'lung':                           'NT=lung;AC=UBERON:0002048',
    'heart':                          'NT=heart;AC=UBERON:0000948',
    'kidney':                         'NT=kidney;AC=UBERON:0002113',
    'pancreas':                       'NT=pancreas;AC=UBERON:0001264',
    'colon':                          'NT=colon;AC=UBERON:0001155',
    'colorectum':                     'NT=colon;AC=UBERON:0001155',
    'prostate':                       'NT=prostate gland;AC=UBERON:0002367',
    'prostate gland':                 'NT=prostate gland;AC=UBERON:0002367',
    'breast':                         'NT=breast;AC=UBERON:0000310',
    'ovary':                          'NT=ovary;AC=UBERON:0000992',
    'ovaries':                        'NT=ovary;AC=UBERON:0000992',
    'spleen':                         'NT=spleen;AC=UBERON:0002106',
    'bone marrow':                    'NT=bone marrow;AC=UBERON:0002371',
    'thymus':                         'NT=thymus;AC=UBERON:0002370',
    'lymph node':                     'NT=lymph node;AC=UBERON:0000029',
    'adipose tissue':                 'NT=adipose tissue;AC=UBERON:0001013',
    'adipose':                        'NT=adipose tissue;AC=UBERON:0001013',
    'skeletal muscle':                'NT=skeletal muscle;AC=UBERON:0001134',
    'muscle':                         'NT=skeletal muscle;AC=UBERON:0001134',
    'skin':                           'NT=skin of body;AC=UBERON:0002097',
    'thyroid':                        'NT=thyroid gland;AC=UBERON:0002046',
    'testis':                         'NT=testis;AC=UBERON:0000473',
    'testes':                         'NT=testis;AC=UBERON:0000473',
    'retina':                         'NT=retina;AC=UBERON:0000966',
    'stomach':                        'NT=stomach;AC=UBERON:0000945',
    'small intestine':                'NT=small intestine;AC=UBERON:0002108',
    'intestine':                      'NT=intestine;AC=UBERON:0000160',
    'endometrium':                    'NT=endometrium;AC=UBERON:0001295',
    'placenta':                       'NT=placenta;AC=UBERON:0001987',
    'aorta':                          'NT=aorta;AC=UBERON:0000947',
    # Cell types as tissue
    'pbmc':                           'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'peripheral blood mononuclear':   'NT=peripheral blood mononuclear cell;AC=CL:0000057',
    'platelet':                       'NT=platelet;AC=CL:0000233',
    'platelets':                      'NT=platelet;AC=CL:0000233',
    'neutrophil':                     'NT=neutrophil;AC=CL:0000775',
    'neutrophils':                    'NT=neutrophil;AC=CL:0000775',
    'exosome':                        'NT=extracellular vesicle;AC=GO:0061695',
    'extracellular vesicle':          'NT=extracellular vesicle;AC=GO:0061695',
}

# ── Organism normalization (NCBI taxonomy) ─────────────────────────────────
ORGANISM_ONT = {
    'homo sapiens': '9606 (Homo sapiens)', 'human': '9606 (Homo sapiens)',
    'humans': '9606 (Homo sapiens)', 'patient': '9606 (Homo sapiens)',
    'mus musculus': '10090 (Mus musculus)', 'mouse': '10090 (Mus musculus)',
    'mice': '10090 (Mus musculus)', 'murine': '10090 (Mus musculus)',
    'rattus norvegicus': '10116 (Rattus norvegicus)', 'rat': '10116 (Rattus norvegicus)',
    'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
    'yeast': '4932 (Saccharomyces cerevisiae)',
    'escherichia coli': '562 (Escherichia coli)', 'e. coli': '562 (Escherichia coli)',
    'e.coli': '562 (Escherichia coli)',
    'drosophila melanogaster': '7227 (Drosophila melanogaster)',
    'fruit fly': '7227 (Drosophila melanogaster)',
    'danio rerio': '7955 (Danio rerio)', 'zebrafish': '7955 (Danio rerio)',
    'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
    'sus scrofa': '9823 (Sus scrofa)', 'pig': '9823 (Sus scrofa)',
    'porcine': '9823 (Sus scrofa)',
    'bos taurus': '9913 (Bos taurus)', 'bovine': '9913 (Bos taurus)',
    'gallus gallus': '9031 (Gallus gallus)', 'chicken': '9031 (Gallus gallus)',
    'caenorhabditis elegans': '6239 (Caenorhabditis elegans)',
    'c. elegans': '6239 (Caenorhabditis elegans)',
    'xenopus laevis': '8355 (Xenopus laevis)',
    'macaca mulatta': '9544 (Macaca mulatta)', 'rhesus': '9544 (Macaca mulatta)',
    'canis lupus familiaris': '9615 (Canis lupus familiaris)', 'dog': '9615 (Canis lupus familiaris)',
}

# ── Cleavage agent — exhaustive synonyms → PSI-MS canonical ───────────────
CLEAVAGE_ONT = {
    # Trypsin variants
    'trypsin/lysc': 'AC=MS:1001251;NT=Trypsin', # often reported together
    'trypsin/lys-c': 'AC=MS:1001251;NT=Trypsin',
    'trypsin': 'AC=MS:1001251;NT=Trypsin',
    'tryps': 'AC=MS:1001251;NT=Trypsin',
    'tryptic': 'AC=MS:1001251;NT=Trypsin',
    'proteomicgrade trypsin': 'AC=MS:1001251;NT=Trypsin',
    'modified trypsin': 'AC=MS:1001251;NT=Trypsin',
    'sequencing grade trypsin': 'AC=MS:1001251;NT=Trypsin',
    # Lys-C variants
    'lys-c': 'AC=MS:1001255;NT=Lys-C',
    'lysc': 'AC=MS:1001255;NT=Lys-C',
    'lys c': 'AC=MS:1001255;NT=Lys-C',
    'lysyl endopeptidase': 'AC=MS:1001255;NT=Lys-C',
    # Glu-C variants
    'glu-c': 'AC=MS:1001917;NT=Glu-C',
    'gluc': 'AC=MS:1001917;NT=Glu-C',
    'endoproteinase glu-c': 'AC=MS:1001917;NT=Glu-C',
    'v8 protease': 'AC=MS:1001917;NT=Glu-C',
    # Other enzymes
    'chymotrypsin': 'AC=MS:1001306;NT=Chymotrypsin',
    'asp-n': 'AC=MS:1001267;NT=Asp-N',
    'aspn': 'AC=MS:1001267;NT=Asp-N',
    'arg-c': 'AC=MS:1001303;NT=Arg-C',
    'argc': 'AC=MS:1001303;NT=Arg-C',
    'cnbr': 'AC=MS:1001325;NT=CNBr',
    'cyanogen bromide': 'AC=MS:1001325;NT=CNBr',
    'elastase': 'AC=MS:1001915;NT=Elastase',
    'pepsin': 'AC=MS:1001940;NT=Pepsin',
    'thermolysin': 'AC=MS:1001942;NT=Thermolysin',
}

# ── Label/quantification → PSI-MS canonical ───────────────────────────────
LABEL_ONT = {
    # Label-free
    'label-free': 'AC=MS:1002038;NT=label free sample',
    'label free': 'AC=MS:1002038;NT=label free sample',
    'lfq': 'AC=MS:1002038;NT=label free sample',
    'unlabeled': 'AC=MS:1002038;NT=label free sample',
    'label_free': 'AC=MS:1002038;NT=label free sample',
    # TMT
    'tmt18plex': 'AC=MS:1003999;NT=TMT18plex',
    'tmt16plex': 'AC=MS:1003998;NT=TMT16plex',
    'tmt pro': 'AC=MS:1003998;NT=TMT16plex',
    'tmtpro': 'AC=MS:1003998;NT=TMT16plex',
    'tmt11plex': 'AC=MS:1002454;NT=TMT11plex',
    'tmt10plex': 'AC=MS:1002454;NT=TMT10plex',
    'tmt10': 'AC=MS:1002454;NT=TMT10plex',
    'tmt6plex': 'AC=MS:1002453;NT=TMT6plex',
    'tmt6': 'AC=MS:1002453;NT=TMT6plex',
    'tmt2plex': 'AC=MS:1002456;NT=TMT2plex',
    'tmt': 'AC=MS:1002453;NT=TMT6plex',  # fallback
    # iTRAQ
    'itraq8plex': 'AC=MS:1002519;NT=iTRAQ8plex',
    'itraq4plex': 'AC=MS:1001985;NT=iTRAQ4plex',
    'itraq': 'AC=MS:1001985;NT=iTRAQ4plex',  # fallback
    # SILAC
    'silac': 'AC=MS:1002791;NT=SILAC',
    'stable isotope labeling': 'AC=MS:1002791;NT=SILAC',
    # Dimethyl
    'dimethyl': 'AC=MS:1002457;NT=Dimethyl',
    'reductive dimethylation': 'AC=MS:1002457;NT=Dimethyl',
    'dimethyl labeling': 'AC=MS:1002457;NT=Dimethyl',
    # Tandem mass tag (synonym)
    'tandem mass tag': 'AC=MS:1002453;NT=TMT6plex',
}

# ── Fragmentation method ───────────────────────────────────────────────────
FRAG_ONT = {
    'hcd': 'AC=MS:1002481;NT=HCD',
    'higher-energy collisional dissociation': 'AC=MS:1002481;NT=HCD',
    'higher energy collisional dissociation': 'AC=MS:1002481;NT=HCD',
    'cid': 'AC=MS:1001880;NT=CID',
    'collision-induced dissociation': 'AC=MS:1001880;NT=CID',
    'collision induced dissociation': 'AC=MS:1001880;NT=CID',
    'etd': 'AC=MS:1001526;NT=ETD',
    'electron transfer dissociation': 'AC=MS:1001526;NT=ETD',
    'ecd': 'AC=MS:1001872;NT=ECD',
    'electron capture dissociation': 'AC=MS:1001872;NT=ECD',
    'uvpd': 'AC=MS:1003246;NT=UVPD',
    'ultraviolet photodissociation': 'AC=MS:1003246;NT=UVPD',
    'ethcd': 'AC=MS:1002686;NT=EThcD',
    'eccid': 'AC=MS:1002686;NT=EThcD',
}

# ── Modification → UNIMOD canonical ───────────────────────────────────────
MOD_ONT = {
    # Fixed modifications
    'carbamidomethyl': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'carbamidomethylation': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'iodoacetamide': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'iaa': 'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'propionamide': 'NT=Propionamide;AC=UNIMOD:24;TA=C;MT=Fixed',
    # Variable modifications
    'oxidation': 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable',
    'oxidized methionine': 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable',
    'methionine oxidation': 'NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable',
    'phosphorylation': 'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable',
    'phospho': 'NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable',
    'serine phosphorylation': 'NT=Phospho;AC=UNIMOD:21;TA=S;MT=Variable',
    'threonine phosphorylation': 'NT=Phospho;AC=UNIMOD:21;TA=T;MT=Variable',
    'tyrosine phosphorylation': 'NT=Phospho;AC=UNIMOD:21;TA=Y;MT=Variable',
    'acetylation': 'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable',
    'lysine acetylation': 'NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable',
    'n-terminal acetylation': 'NT=Acetyl;AC=UNIMOD:1;TA=N-term;MT=Variable',
    'ubiquitination': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'ubiquitinylation': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'diglycine': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'di-glycine': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'gg remnant': 'NT=GlyGly;AC=UNIMOD:121;TA=K;MT=Variable',
    'methylation': 'NT=Methyl;AC=UNIMOD:34;TA=K,R;MT=Variable',
    'deamidation': 'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable',
    'deamidated': 'NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable',
    'sumoylation': 'NT=SUMO;AC=UNIMOD:3;TA=K;MT=Variable',
    'succinylation': 'NT=Succinyl;AC=UNIMOD:64;TA=K;MT=Variable',
    'crotonylation': 'NT=Crotonyl;AC=UNIMOD:1363;TA=K;MT=Variable',
    'glutarylation': 'NT=Glutaryl;AC=UNIMOD:1848;TA=K;MT=Variable',
    'malonylation': 'NT=Malonyl;AC=UNIMOD:747;TA=K;MT=Variable',
    'formylation': 'NT=Formyl;AC=UNIMOD:122;TA=K;MT=Variable',
    'hydroxylation': 'NT=Hydroxyl;AC=UNIMOD:35;TA=P;MT=Variable',
    'proline hydroxylation': 'NT=Hydroxyl;AC=UNIMOD:35;TA=P;MT=Variable',
    'nitrosylation': 'NT=Nitrosyl;AC=UNIMOD:275;TA=C;MT=Variable',
    's-nitrosylation': 'NT=Nitrosyl;AC=UNIMOD:275;TA=C;MT=Variable',
    'neddylation': 'NT=NEDD;AC=UNIMOD:1;TA=K;MT=Variable',
    'palmitoylation': 'NT=Palmitoyl;AC=UNIMOD:47;TA=C;MT=Variable',
    'myristoylation': 'NT=Myristoyl;AC=UNIMOD:45;TA=G;MT=Variable',
    'tmt6plex': 'NT=TMT6plex;AC=UNIMOD:737;TA=K;MT=Fixed',
    'tmt label': 'NT=TMT6plex;AC=UNIMOD:737;TA=K;MT=Fixed',
}

# ── Reduction reagent ──────────────────────────────────────────────────────
REDUCTION_ONT = {
    'dtt': 'AC=MS:1000578;NT=DTT',
    'dithiothreitol': 'AC=MS:1000578;NT=DTT',
    'tcep': 'AC=MS:1001135;NT=TCEP',
    'tris(2-carboxyethyl)phosphine': 'AC=MS:1001135;NT=TCEP',
    'beta-mercaptoethanol': 'AC=MS:1000382;NT=beta-mercaptoethanol',
    'bme': 'AC=MS:1000382;NT=beta-mercaptoethanol',
    '2-mercaptoethanol': 'AC=MS:1000382;NT=beta-mercaptoethanol',
}

# ── Alkylation reagent ─────────────────────────────────────────────────────
ALKYLATION_ONT = {
    'iodoacetamide': 'AC=PRIDE:0000126;NT=Iodoacetamide',
    'iaa': 'AC=PRIDE:0000126;NT=Iodoacetamide',
    'chloroacetamide': 'AC=PRIDE:0000126;NT=Chloroacetamide',
    'caa': 'AC=PRIDE:0000126;NT=Chloroacetamide',
    'n-ethylmaleimide': 'AC=PRIDE:0000459;NT=N-ethylmaleimide',
    'nem': 'AC=PRIDE:0000459;NT=N-ethylmaleimide',
    '4-vinylpyridine': 'AC=PRIDE:0000101;NT=4-vinylpyridine',
    'iodoacetic acid': 'AC=PRIDE:0000127;NT=Iodoacetic acid',
}

print('Ontology dictionaries loaded.')
print(f'  Instruments : {len(INSTRUMENT_ONT)} entries')
print(f'  Tissues     : {len(TISSUE_ONT)} entries')
print(f'  Organisms   : {len(ORGANISM_ONT)} entries')
print(f'  Cleavage    : {len(CLEAVAGE_ONT)} entries')
print(f'  Labels      : {len(LABEL_ONT)} entries')
print(f'  Fragments   : {len(FRAG_ONT)} entries')
print(f'  Mods        : {len(MOD_ONT)} entries')

Ontology dictionaries loaded.
  Instruments : 30 entries
  Tissues     : 65 entries
  Organisms   : 34 entries
  Cleavage    : 26 entries
  Labels      : 25 entries
  Fragments   : 14 entries
  Mods        : 39 entries


## 2. Normalization Functions

In [3]:
def normalize(raw, ontology, default=None):
    """Map a raw string to canonical ontology term.
    Tries longest match first to avoid 'trypsin' matching before 'trypsin/lys-c'.
    """
    if not raw or str(raw).strip().lower() in ('not applicable', 'na', 'n/a', ''):
        return default
    s = str(raw).lower().strip()
    # Try longest key first
    for key in sorted(ontology.keys(), key=len, reverse=True):
        if key in s:
            return ontology[key]
    return default

def normalize_instrument(raw):
    """Normalize instrument string, also handles NT=/AC= compound strings."""
    if not raw or str(raw).strip().lower() in ('not applicable', 'na', 'n/a', ''):
        return None
    s = str(raw)
    # Already normalized: extract NT= and AC= and reorder
    ac = re.search(r'AC=(MS:\d+)', s)
    nt = re.search(r'NT=([^;]+)', s)
    if ac and nt:
        nt_clean = nt.group(1).strip()
        ac_clean = ac.group(1).strip()
        return f'AC={ac_clean};NT={nt_clean}'
    # Try ontology lookup
    return normalize(raw, INSTRUMENT_ONT)

def normalize_label_from_filename(raw_file):
    """Extract label information from raw filename."""
    rf = str(raw_file).upper()
    rf_low = str(raw_file).lower()

    # TMT — check for plex number first
    m = re.search(r'TMT(PRO|18|16|11|10|6|2)', rf)
    if m:
        plex = m.group(1)
        plex_map = {'PRO':'16','18':'18','16':'16','11':'11','10':'10','6':'6','2':'2'}
        p = plex_map.get(plex, '6')
        acc = {'18':'MS:1003999','16':'MS:1003998','11':'MS:1002454',
               '10':'MS:1002454','6':'MS:1002453','2':'MS:1002456'}
        return f'AC={acc[p]};NT=TMT{p}plex'
    if 'TMT' in rf:
        return 'AC=MS:1002453;NT=TMT6plex'

    # iTRAQ
    m = re.search(r'ITRAQ(8|4)', rf)
    if m:
        p = m.group(1)
        return f"AC={'MS:1002519' if p=='8' else 'MS:1001985'};NT=iTRAQ{p}plex"

    # SILAC
    if re.search(r'SILAC|_H_|_HVY|_HEAVY|_L_|_LGT|_LIGHT|_M_|_MED', rf):
        return 'AC=MS:1002791;NT=SILAC'

    # Label-free
    if re.search(r'LFQ|LABELFREE|LABEL.FREE|_LF_|_LF\d', rf):
        return 'AC=MS:1002038;NT=label free sample'

    return None

def parse_fraction(raw_file):
    """Extract fraction identifier from filename."""
    rf = str(raw_file)
    patterns = [
        r'[_\-]f(\d{1,3})[_\-\.]',           # _f01_ or -f1-
        r'[_\-]fr(\d{1,3})[_\-\.]',          # _fr01_
        r'[_\-]frac(?:tion)?[_\-]?(\d{1,3})', # _frac1_ or _fraction01
        r'[_\-]scx(\d{1,3})[_\-\.]',         # SCX fraction
        r'[_\-]hprp(\d{1,3})[_\-\.]',        # hpRP fraction
        r'[_\-](\d{1,3})of\d+[_\-\.]',      # _1of24_
        r'fraction(\d{1,3})',                    # fraction12
    ]
    for p in patterns:
        m = re.search(p, rf, re.I)
        if m:
            val = m.group(1)
            if val.isdigit() and 1 <= int(val) <= 200:  # sanity check
                return str(int(val))  # strip leading zeros
    return None

def parse_biol_replicate(raw_file):
    """Extract biological replicate number from filename."""
    rf = str(raw_file)
    patterns = [
        r'[_\-]biol?rep[_\-]?(\d+)',         # _biorep1_ or _biolrep01
        r'[_\-]br(\d+)[_\-\.]',             # _br1_
        r'[_\-]rep(\d+)[_\-\.]',            # _rep1_
        r'[_\-]r(\d{1,2})[_\-\.]',         # _r1_ (short form)
        r'replicate[_\-]?(\d+)',              # replicate1
    ]
    for p in patterns:
        m = re.search(p, rf, re.I)
        if m:
            val = m.group(1)
            if val.isdigit() and 1 <= int(val) <= 50:
                return str(int(val))
    return None

def parse_tech_replicate(raw_file):
    """Extract technical replicate number from filename."""
    rf = str(raw_file)
    patterns = [
        r'[_\-]techrep[_\-]?(\d+)',
        r'[_\-]tr(\d+)[_\-\.]',
        r'[_\-]inj[_\-]?(\d+)',
        r'[_\-]inject(?:ion)?[_\-]?(\d+)',
        r'[_\-]technical[_\-](\d+)',
    ]
    for p in patterns:
        m = re.search(p, rf, re.I)
        if m:
            val = m.group(1)
            if val.isdigit() and 1 <= int(val) <= 20:
                return str(int(val))
    return None

def parse_condition_from_filename(raw_file):
    """Extract experimental condition tokens — disease, genotype, treatment."""
    rf = str(raw_file)
    rf_up = rf.upper()
    rf_low = rf.lower()
    conditions = {}

    # Genotype markers
    if re.search(r'[_\-\.](WT|WILDTYPE|WILD.TYPE)[_\-\.]', rf_up):
        conditions['Characteristics[Genotype]'] = 'wild-type'
    elif re.search(r'[_\-\.](KO|KNOCKOUT|KNOCK.OUT)[_\-\.]', rf_up):
        conditions['Characteristics[Genotype]'] = 'knockout'
    elif re.search(r'[_\-\.](KI|KNOCKIN|KNOCK.IN)[_\-\.]', rf_up):
        conditions['Characteristics[Genotype]'] = 'knock-in'
    elif re.search(r'[_\-\.](HET|HETEROZYGOUS)[_\-\.]', rf_up):
        conditions['Characteristics[Genotype]'] = 'heterozygous'

    # Sample type from filename prefix tokens
    if re.search(r'[_\-\.]?(AD|ALZ)[_\-\.]', rf_up):
        conditions['Characteristics[Disease]'] = 'Alzheimer disease'
    if re.search(r'[_\-\.]?(CTRL|CTL|CONTROL|HC|HEALTHY)[_\-\.]', rf_up):
        conditions['Characteristics[Disease]'] = 'normal'
    if re.search(r'[_\-\.]?(PD|PARK)[_\-\.]', rf_up):
        conditions['Characteristics[Disease]'] = 'Parkinson disease'
    if re.search(r'[_\-\.]?(COVID|COV|SARS)[_\-\.]', rf_up):
        conditions['Characteristics[Disease]'] = 'COVID-19'

    # SILAC channel
    if re.search(r'[_\-\.](HEAVY|HVY|_H_)[_\-\.]', rf_up):
        conditions['Characteristics[Label]'] = 'heavy'
    elif re.search(r'[_\-\.](LIGHT|LGT|_L_)[_\-\.]', rf_up):
        conditions['Characteristics[Label]'] = 'light'
    elif re.search(r'[_\-\.](MED|MEDIUM|_M_)[_\-\.]', rf_up):
        conditions['Characteristics[Label]'] = 'medium'

    # Sample tissue hint
    if re.search(r'[_\-\.](PL|PLASMA)[_\-\.]', rf_up):
        conditions['Characteristics[OrganismPart]'] = 'NT=blood plasma;AC=UBERON:0001969'
    elif re.search(r'[_\-\.](SER|SERUM)[_\-\.]', rf_up):
        conditions['Characteristics[OrganismPart]'] = 'NT=blood serum;AC=UBERON:0001977'
    elif re.search(r'[_\-\.](CSF)[_\-\.]', rf_up):
        conditions['Characteristics[OrganismPart]'] = 'NT=cerebrospinal fluid;AC=UBERON:0001359'
    elif re.search(r'[_\-\.](URINE|UR)[_\-\.]', rf_up):
        conditions['Characteristics[OrganismPart]'] = 'NT=urine;AC=UBERON:0001088'

    return conditions

print('Normalization and filename parsing functions defined.')
# Quick tests
print(f"  normalize_instrument('Q Exactive HF') = {normalize_instrument('Q Exactive HF')}")
print(f"  normalize('trypsin/lysc', CLEAVAGE_ONT) = {normalize('trypsin/lysc', CLEAVAGE_ONT)}")
print(f"  parse_fraction('PXD001_f12_rep1.raw') = {parse_fraction('PXD001_f12_rep1.raw')}")
print(f"  parse_biol_replicate('sample_r3_technical.raw') = {parse_biol_replicate('sample_r3_technical.raw')}")
print(f"  parse_condition_from_filename('AD_pl01.raw') = {parse_condition_from_filename('AD_pl01.raw')}")

Normalization and filename parsing functions defined.
  normalize_instrument('Q Exactive HF') = AC=MS:1002523;NT=Q Exactive HF
  normalize('trypsin/lysc', CLEAVAGE_ONT) = AC=MS:1001251;NT=Trypsin
  parse_fraction('PXD001_f12_rep1.raw') = 12
  parse_biol_replicate('sample_r3_technical.raw') = 3
  parse_condition_from_filename('AD_pl01.raw') = {'Characteristics[Disease]': 'Alzheimer disease'}


## 3. Load training data

In [4]:
sample_sub  = pd.read_csv(SAMPLE_SUB)
id_cols     = ['ID', 'PXD', 'Raw Data File', 'Usage']
target_cols = [c for c in sample_sub.columns
               if c not in id_cols and 'Unnamed' not in c]
all_base    = set([re.sub(r'\.\d+$', '', c) for c in target_cols])

def _strip_wrapper(col):
    """'Characteristics[Organism]' → 'Organism', 'Comment[Instrument]' → 'Instrument'"""
    m = re.match(r'(?:characteristics|comment|factor value)\[(.+?)\]', col, re.I)
    return m.group(1) if m else col

def _find_col(col, df_cols):
    """Match target col against a DataFrame's actual column names."""
    if col in df_cols: return col
    base = re.sub(r'\.\d+$', '', col)
    if base in df_cols: return base
    stripped = _strip_wrapper(base)
    if stripped in df_cols: return stripped
    return None

# Build vocabulary and majority modes from training
col_counters = {col: Counter() for col in target_cols}
col_vocab    = defaultdict(set)
train_files  = sorted(TRAIN_SDRF_DIR.glob('*.csv')) if TRAIN_SDRF_DIR.exists() else []

# Also try .tsv if no csv found
if not train_files:
    train_files = sorted(TRAIN_SDRF_DIR.parent.glob('**/*.tsv'))
    if not train_files:
        # Check alternate local path
        alt = Path.cwd().parent / 'data' / 'TrainingSDRFs'
        if alt.exists():
            train_files = sorted(alt.glob('*.tsv')) + sorted(alt.glob('*.csv'))

for fp in train_files:
    sep = '\t' if fp.suffix == '.tsv' else ','
    try:
        df = pd.read_csv(fp, low_memory=False, sep=sep)
    except Exception:
        continue
    df_cols = set(df.columns)
    for col in target_cols:
        base = re.sub(r'\.\d+$', '', col)
        match_col = _find_col(col, df_cols)
        if match_col:
            vals = df[match_col].dropna().astype(str)
            vals = vals[~vals.isin(['Not Applicable','not applicable','NA','nan'])]
            col_counters[col].update(vals.tolist())
            col_vocab[base].update(vals.tolist())

global_modes = {}
non_na_ratio = {}
n_train = max(len(train_files), 1)
for col in target_cols:
    total = sum(col_counters[col].values())
    if total > 0:
        global_modes[col] = col_counters[col].most_common(1)[0][0]
        non_na_ratio[col] = total / n_train
    else:
        global_modes[col] = 'Not Applicable'
        non_na_ratio[col] = 0.0

# Per-PXD training lookup
train_pxd_sdrf = {}
for fp in train_files:
    pxd = fp.stem.replace('Harmonized_','').replace('_cleaned.sdrf','').split('.')[0]
    sep = '\t' if fp.suffix == '.tsv' else ','
    try:
        df = pd.read_csv(fp, low_memory=False, sep=sep)
    except Exception:
        continue
    df_cols = set(df.columns)
    pxd_vals = {}
    for col in target_cols:
        m = _find_col(col, df_cols)
        if m:
            vals = df[m].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','n/a','na',''])]
            uniq = list(vals.unique())
            if uniq:
                pxd_vals[col] = uniq
    train_pxd_sdrf[pxd] = pxd_vals

print(f'Training SDRFs   : {len(train_files)}')
print(f'Target columns   : {len(target_cols)}')
print(f'Vocab terms      : {sum(len(v) for v in col_vocab.values()):,}')


Training SDRFs   : 103
Target columns   : 77
Vocab terms      : 1,538


In [5]:
SECTIONS = ['TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
            'EXPERIMENTAL','SAMPLE PREPARATION','MASS SPECTROMETRY',
            'LC-MS','PROTEIN DIGESTION','DATA ACQUISITION','CELL CULTURE']
METHOD_KWS = ['method','material','protocol','digest','spectr','chromat',
              'prep','enrichment','culture','experimental']

def get_text(pub_dict, max_chars=None):
    parts = []
    for key in SECTIONS:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    for key, val in pub_dict.items():
        if key.upper() in SECTIONS: continue
        if any(kw in key.lower() for kw in METHOD_KWS):
            if isinstance(val, list): val = ' '.join(str(x) for x in val)
            if val.strip(): parts.append(val.strip())
    for key in ['ABSTRACT','TITLE']:
        val = pub_dict.get(key, '')
        if isinstance(val, list): val = ' '.join(str(x) for x in val)
        if val.strip(): parts.append(val.strip())
    text = ' '.join(parts)
    return text[:max_chars] if max_chars else text

test_docs = {}
if TEST_TEXT_DIR.exists():
    for fp in sorted(TEST_TEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: test_docs[pxd] = d
        except Exception:
            pass

print(f'Test papers loaded: {len(test_docs)}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

Test papers loaded: 16
  PubText: 0 chars
  PXD004010: 11,237 chars
  PXD016436: 7,509 chars
  PXD019519: 45,042 chars
  PXD025663: 20,717 chars
  PXD040582: 19,714 chars
  PXD050621: 14,085 chars
  PXD061009: 34,629 chars
  PXD061090: 19,600 chars
  PXD061136: 15,870 chars
  PXD061195: 9,808 chars
  PXD061285: 34,103 chars
  PXD062014: 29,451 chars
  PXD062469: 16,510 chars
  PXD062877: 35,268 chars
  PXD064564: 19,510 chars


In [6]:
http_session = requests.Session()
http_session.headers.update({'User-Agent': 'SDRF-Extractor/2.0'})

def fetch_pride(pxd):
    try:
        r = http_session.get(
            f'https://www.ebi.ac.uk/pride/ws/archive/v2/projects/{pxd}',
            timeout=PRIDE_TIMEOUT
        )
        if r.status_code != 200: return {}
        d = r.json()
        out = defaultdict(list)

        for o in d.get('organisms', []):
            name = o.get('name','')
            if name:
                norm = normalize(name, ORGANISM_ONT)
                out['Characteristics[Organism]'].append(norm or name.strip())

        for op in (d.get('organisms_part') or d.get('tissues') or []):
            name = op.get('name',''); acc = op.get('accession','')
            if name and name.lower() not in ('not available','n/a',''):
                # Try ontology first
                norm = normalize(name, TISSUE_ONT)
                if norm:
                    out['Characteristics[OrganismPart]'].append(norm)
                elif acc:
                    out['Characteristics[OrganismPart]'].append(f'NT={name};AC={acc}')
                else:
                    out['Characteristics[OrganismPart]'].append(f'NT={name}')

        for dis in d.get('diseases',[]):
            name = dis.get('name','')
            if name and name.lower() not in ('not available','n/a','none','normal',''):
                out['Characteristics[Disease]'].append(name)

        for inst in d.get('instruments',[]):
            name = inst.get('name',''); acc = inst.get('accession','')
            if name:
                norm = normalize_instrument(name)
                if norm:
                    out['Comment[Instrument]'].append(norm)
                elif acc:
                    out['Comment[Instrument]'].append(f'AC={acc};NT={name}')

        for qm in d.get('quantification_methods',[]):
            name = qm.get('name','')
            if name:
                norm = normalize(name, LABEL_ONT)
                out['Characteristics[Label]'].append(norm or name)

        return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}
    except Exception as e:
        print(f'  PRIDE error {pxd}: {e}')
        return {}

def fetch_px_xml(pxd):
    out = defaultdict(list)
    try:
        r = http_session.get(
            f'https://proteomecentral.proteomexchange.org/cgi/GetDataset'
            f'?ID={pxd}&outputMode=XML&test=no',
            timeout=PX_TIMEOUT
        )
        if r.status_code != 200: return {}
        xml = r.text
        for m in re.finditer(r'<cvParam[^>]+accession="(MS:\d+)"[^>]+name="([^"]+)"', xml):
            if 'instrument' in m.group(2).lower():
                norm = normalize_instrument(m.group(2))
                if norm: out['Comment[Instrument]'].append(norm)
        for m in re.finditer(r'<cvParam[^>]+accession="(NEWT:\d+)"[^>]+name="([^"]+)"', xml):
            tax = m.group(1).replace('NEWT:',''); name = m.group(2)
            norm = normalize(name, ORGANISM_ONT)
            out['Characteristics[Organism]'].append(norm if norm else f'{tax} ({name})')
    except Exception as e:
        print(f'  PX XML error {pxd}: {e}')
    return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}

print('API fetchers defined.')

API fetchers defined.


In [7]:
_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'
_CLINICAL = re.compile(
    r'\b(patient|cohort|biopsy|tumor|tumour|cancer|carcinoma|malignant|'
    r'diagnosed|clinical|disease|healthy\s+(?:control|donor)|specimen|'
    r'hospital|surgical|resection|case|control)\b', re.I
)

def regex_extract(pub_dict):
    """Extract protocol parameters from paper text with full ontology normalization."""
    text     = get_text(pub_dict)
    text_low = text.lower()
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]:
            out[col].append(val)

    # ── Organism (normalized) ────────────────────────────────────────────
    for pat, key in [
        (re.compile(r'\b(homo\s+sapiens|human(?:s)?)\b', re.I), 'human'),
        (re.compile(r'\b(mus\s+musculus|mouse|mice|murine)\b', re.I), 'mouse'),
        (re.compile(r'\b(rattus\s+norvegicus|rat(?:s)?)\b', re.I), 'rat'),
        (re.compile(r'\b(saccharomyces\s+cerevisiae|(?<!\w)yeast)\b', re.I), 'yeast'),
        (re.compile(r'\b(escherichia\s+coli|e\.?\s*coli)\b', re.I), 'e. coli'),
        (re.compile(r'\b(danio\s+rerio|zebrafish)\b', re.I), 'zebrafish'),
        (re.compile(r'\b(drosophila\s+melanogaster)\b', re.I), 'drosophila melanogaster'),
        (re.compile(r'\b(sus\s+scrofa|porcine)\b', re.I), 'pig'),
        (re.compile(r'\b(bos\s+taurus|bovine)\b', re.I), 'bovine'),
        (re.compile(r'\b(gallus\s+gallus|chicken)\b', re.I), 'chicken'),
        (re.compile(r'\b(arabidopsis\s+thaliana)\b', re.I), 'arabidopsis thaliana'),
        (re.compile(r'\b(caenorhabditis\s+elegans|c\.\s*elegans)\b', re.I), 'c. elegans'),
        (re.compile(r'\b(macaca\s+mulatta|rhesus\s+macaque)\b', re.I), 'macaca mulatta'),
    ]:
        if pat.search(text):
            norm = ORGANISM_ONT.get(key)
            if norm: add('Characteristics[Organism]', norm)

    # ── OrganismPart (longest match, normalized) ─────────────────────────
    for tissue in sorted(TISSUE_ONT.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(tissue) + r'\b', text_low):
            add('Characteristics[OrganismPart]', TISSUE_ONT[tissue])

    # ── CleavageAgent (longest match, normalized) ────────────────────────
    for enzyme in sorted(CLEAVAGE_ONT.keys(), key=len, reverse=True):
        pat = re.compile(_NEG + r'\b' + re.escape(enzyme) + r'\b', re.I)
        if pat.search(text_low):
            add('Characteristics[CleavageAgent]', CLEAVAGE_ONT[enzyme])
            break  # take first/best match

    # ── Label (normalized) ───────────────────────────────────────────────
    for label_key in sorted(LABEL_ONT.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(label_key) + r'\b', text_low):
            add('Characteristics[Label]', LABEL_ONT[label_key])
            break

    # ── Reduction reagent ────────────────────────────────────────────────
    for key in sorted(REDUCTION_ONT.keys(), key=len, reverse=True):
        pat = re.compile(_NEG + r'\b' + re.escape(key) + r'\b', re.I)
        if pat.search(text_low):
            add('Characteristics[ReductionReagent]', REDUCTION_ONT[key])
            break

    # ── Alkylation reagent ───────────────────────────────────────────────
    for key in sorted(ALKYLATION_ONT.keys(), key=len, reverse=True):
        pat = re.compile(r'\b' + re.escape(key) + r'\b', re.I)
        if pat.search(text_low):
            add('Characteristics[AlkylationReagent]', ALKYLATION_ONT[key])
            break

    # ── Modifications ────────────────────────────────────────────────────
    for key in sorted(MOD_ONT.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(key) + r'\b', text_low):
            add('Characteristics[Modification]', MOD_ONT[key])

    # ── Instrument ───────────────────────────────────────────────────────
    for key in sorted(INSTRUMENT_ONT.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(key) + r'\b', text_low):
            add('Comment[Instrument]', INSTRUMENT_ONT[key])
            break

    # ── Fragmentation ────────────────────────────────────────────────────
    for key in sorted(FRAG_ONT.keys(), key=len, reverse=True):
        if re.search(r'\b' + re.escape(key) + r'\b', text_low):
            add('Comment[FragmentationMethod]', FRAG_ONT[key])

    # ── AcquisitionMethod ────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(dda|data[\s\-]dependent)\b', re.I), 'AC=MS:1003215;NT=DDA'),
        (re.compile(r'\b(dia|data[\s\-]independent|swath)\b', re.I), 'AC=MS:1003215;NT=DIA'),
        (re.compile(r'\b(prm|parallel\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=PRM'),
        (re.compile(r'\b(srm|mrm|selected\s+reaction\s+monitoring)\b', re.I), 'AC=MS:1001501;NT=SRM'),
    ]:
        if pat.search(text):
            add('Comment[AcquisitionMethod]', val); break

    # ── IonizationType ───────────────────────────────────────────────────
    if re.search(r'\b(nano[\s\-]?esi|nesi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000398;NT=nanoESI')
    elif re.search(r'\b(electrospray|esi)\b', text, re.I):
        add('Comment[IonizationType]', 'AC=MS:1000073;NT=ESI')

    # ── MS2MassAnalyzer ──────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(orbitrap)\b', re.I), 'AC=MS:1000484;NT=Orbitrap'),
        (re.compile(r'\b(ion\s*trap)\b', re.I), 'AC=MS:1000264;NT=ion trap'),
        (re.compile(r'\b(tof)\b', re.I), 'AC=MS:1000084;NT=TOF'),
    ]:
        if pat.search(text):
            add('Comment[MS2MassAnalyzer]', val); break

    # ── Sex ──────────────────────────────────────────────────────────────
    if re.search(r'\b(male\s+and\s+female|both\s+sexes|mixed\s+sex)\b', text, re.I):
        add('Characteristics[Sex]', 'male and female')
    elif re.search(r'\b(male\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'male')
    elif re.search(r'\b(female\s+(?:donors?|subjects?|patients?|mice|rats?))\b', text, re.I):
        add('Characteristics[Sex]', 'female')

    # ── Disease (clinical context gated) ─────────────────────────────────
    if _CLINICAL.search(text):
        for pat, val in [
            (re.compile(r'\b(alzheimer[\s\']?s?\s+disease|ad\s+(?:patients?|brains?))\b', re.I), 'Alzheimer disease'),
            (re.compile(r'\b(parkinson[\s\']?s?\s+disease)\b', re.I), 'Parkinson disease'),
            (re.compile(r'\b(type\s+2\s+diabetes(?:\s+mellitus)?|t2d(?:m)?)\b', re.I), 'type 2 diabetes mellitus'),
            (re.compile(r'\b(breast\s+(?:cancer|carcinoma))\b', re.I), 'breast carcinoma'),
            (re.compile(r'\b(colorectal\s+(?:cancer|carcinoma)|colon\s+cancer)\b', re.I), 'colorectal carcinoma'),
            (re.compile(r'\b(non[\s\-]small[\s\-]cell\s+lung|nsclc)\b', re.I), 'non-small cell lung carcinoma'),
            (re.compile(r'\b(lung\s+(?:cancer|carcinoma|adenocarcinoma))\b', re.I), 'lung carcinoma'),
            (re.compile(r'\b(glioblastoma|gbm)\b', re.I), 'glioblastoma'),
            (re.compile(r'\b(melanoma)\b', re.I), 'melanoma'),
            (re.compile(r'\b(prostate\s+(?:cancer|carcinoma))\b', re.I), 'prostate carcinoma'),
            (re.compile(r'\b(ovarian\s+(?:cancer|carcinoma))\b', re.I), 'ovarian carcinoma'),
            (re.compile(r'\b(hepatocellular\s+carcinoma|hcc)\b', re.I), 'hepatocellular carcinoma'),
            (re.compile(r'\b(pancreatic\s+(?:cancer|ductal\s+adenocarcinoma)|pdac)\b', re.I), 'pancreatic ductal adenocarcinoma'),
            (re.compile(r'\b(multiple\s+myeloma)\b', re.I), 'multiple myeloma'),
            (re.compile(r'\b(acute\s+myeloid\s+leukemia|aml)\b', re.I), 'acute myeloid leukemia'),
            (re.compile(r'\b(covid[\s\-]?19|sars[\s\-]?cov[\s\-]?2)\b', re.I), 'COVID-19'),
            (re.compile(r'\b(osteoarthritis)\b', re.I), 'osteoarthritis'),
            (re.compile(r'\b(rheumatoid\s+arthritis)\b', re.I), 'rheumatoid arthritis'),
            (re.compile(r'\b(amyotrophic\s+lateral\s+sclerosis|als)\b', re.I), 'amyotrophic lateral sclerosis'),
            (re.compile(r'\b(healthy\s+(?:controls?|donors?|volunteers?))\b', re.I), 'normal'),
        ]:
            if pat.search(text):
                add('Characteristics[Disease]', val)

    # ── Strain ───────────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(C57BL/6J?)\b'), 'C57BL/6J'),
        (re.compile(r'\b(BALB/c)\b'), 'BALB/c'),
        (re.compile(r'\b(Sprague[\s\-]Dawley)\b', re.I), 'Sprague-Dawley'),
        (re.compile(r'\b(Wistar)\b', re.I), 'Wistar'),
        (re.compile(r'\b(NOD/SCID)\b', re.I), 'NOD/SCID'),
        (re.compile(r'\b(ob/ob)\b'), 'ob/ob'),
        (re.compile(r'\b(FVB/N)\b'), 'FVB/N'),
    ]:
        m = pat.search(text)
        if m:
            add('Characteristics[Strain]', val); break

    # ── Genotype ─────────────────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(wild[\s\-]?type|wt(?:\s+cells?|\s+mice)?)\b', re.I), 'wild-type'),
        (re.compile(r'\b(knockout|knock[\s\-]out|ko(?:\s+cells?|\s+mice)?)\b', re.I), 'knockout'),
        (re.compile(r'\b(transgenic)\b', re.I), 'transgenic'),
        (re.compile(r'\b(heterozygous)\b', re.I), 'heterozygous'),
        (re.compile(r'\b(homozygous)\b', re.I), 'homozygous'),
    ]:
        if pat.search(text):
            add('Characteristics[Genotype]', val); break

    # ── DevelopmentalStage ───────────────────────────────────────────────
    for pat, val in [
        (re.compile(r'\b(adult(?:s)?)\b', re.I), 'adult'),
        (re.compile(r'\b(embryo(?:nic)?)\b', re.I), 'embryo'),
        (re.compile(r'\b(fetal|fetus|foetal)\b', re.I), 'fetal'),
        (re.compile(r'\b(neonatal|newborn)\b', re.I), 'neonatal'),
        (re.compile(r'\b(juvenile|adolescent)\b', re.I), 'juvenile'),
    ]:
        if pat.search(text):
            add('Characteristics[DevelopmentalStage]', val); break

    # ── Numeric protocol fields ──────────────────────────────────────────
    for pat in [
        re.compile(r'(\d+)[\s\-]min(?:ute)?\s+(?:gradient|linear\s+gradient)\b', re.I),
        re.compile(r'gradient\s+(?:of\s+)?(\d+)[\s\-]?min\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[GradientTime]', f'{m.group(1)} min'); break

    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL|µl|µL|ul|uL)\s*/\s*min', text)
    if m:
        unit = 'nL' if m.group(2).lower() == 'nl' else 'µL'
        add('Comment[FlowRateChromatogram]', f'{m.group(1)} {unit}/min')

    for pat in [
        re.compile(r'(?:precursor|ms1)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*ppm\s+(?:for\s+)?(?:precursor|ms1)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'ppm'
            add('Comment[PrecursorMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:fragment|ms2)\s+(?:mass\s+)?tolerance(?:\s+of)?\s+(\d+(?:\.\d+)?)\s*(ppm|da|mda)', re.I),
        re.compile(r'(\d+(?:\.\d+)?)\s*(da|mda)\s+(?:for\s+)?(?:fragment|ms2)', re.I),
    ]:
        m = pat.search(text)
        if m:
            unit = m.group(2) if m.lastindex and m.lastindex >= 2 else 'Da'
            add('Comment[FragmentMassTolerance]', f'{m.group(1)} {unit}'); break

    for pat in [
        re.compile(r'(?:up\s+to\s+|allowing\s+(?:up\s+to\s+)?)(\d)\s+missed\s+cleavages?', re.I),
        re.compile(r'missed\s+cleavages?\s*[=:≤]\s*(\d)', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[NumberOfMissedCleavages]', m.group(1)); break

    for pat in [
        re.compile(r'(\d+)\s+(?:independent\s+)?biological\s+replicates?', re.I),
        re.compile(r'performed\s+in\s+(triplicate|duplicate|quadruplicate)\b', re.I),
    ]:
        m = pat.search(text)
        if m:
            wm = {'triplicate':'3','duplicate':'2','quadruplicate':'4'}
            val = wm.get(m.group(1).lower(), m.group(1)) if m.lastindex else '3'
            add('Characteristics[NumberOfBiologicalReplicates]', val); break

    for pat in [
        re.compile(r'(?:fractionated\s+into|divided\s+into)\s+(\d+)\s+fractions?', re.I),
        re.compile(r'(\d+)\s+(?:scx|hprp|rp)?\s*fractions?\s+(?:were|of)', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Comment[NumberOfFractions]', m.group(1)); break

    for pat in [
        re.compile(r'cohort\s+of\s+(\d+)\s+(?:patients?|subjects?)', re.I),
        re.compile(r'total\s+of\s+(\d+)\s+samples?', re.I),
    ]:
        m = pat.search(text)
        if m:
            add('Characteristics[NumberOfSamples]', m.group(1)); break

    # MaterialType inference
    if 'Characteristics[CellLine]' in out or re.search(r'\b(hela|hek|jurkat|mcf7|a549)\b', text_low):
        add('Characteristics[MaterialType]', 'cell line')
    elif re.search(r'\b(tissue(?:s)?(?!\s+culture)|biopsy|tumor)\b', text, re.I):
        add('Characteristics[MaterialType]', 'tissue')
    elif re.search(r'\b(plasma|serum|urine|csf|whole\s+blood)\b', text, re.I):
        add('Characteristics[MaterialType]', 'biofluid')

    # Cap multi-value columns
    for col in ['Characteristics[OrganismPart]','Characteristics[Modification]','Characteristics[Disease]']:
        if col in out:
            out[col] = out[col][:4]

    return dict(out)

print('Regex extractor defined.')

Regex extractor defined.


## 4. Main Pipeline — Per-File Assignment

This is the key architectural difference from v1. For each PXD:
1. Pull PXD-level data from PRIDE + regex (same for all files in dataset)
2. Parse each raw filename for per-file metadata (different per file)
3. Merge: per-file overrides PXD-level where conflict exists
4. Normalize everything through ontologies before writing

In [8]:
from tqdm import tqdm

final_sub = pd.read_csv(SAMPLE_SUB).copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

for pxd, pxd_df in tqdm(final_sub.groupby('PXD'), desc='Extracting PXDs'):
    idx       = pxd_df.index
    raw_files = pxd_df['Raw Data File'].tolist()
    pub_dict  = test_docs.get(pxd, {})

    # ── PXD-level extraction (same for all files in this dataset) ────────
    pxd_vals = defaultdict(list)

    def pxd_add(col, val):
        if val and str(val).strip().lower() not in ('not applicable','na','n/a',''):
            if val not in pxd_vals[col]:
                pxd_vals[col].append(str(val).strip())

    def pxd_add_list(col, vals):
        for v in (vals or []): pxd_add(col, v)

    # Layer 0: training overlap
    if pxd in train_pxd_sdrf:
        for col, vals in train_pxd_sdrf[pxd].items():
            pxd_add_list(col, vals)

    # Layer 1: PRIDE API
    for col, vals in fetch_pride(pxd).items():
        pxd_add_list(col, vals)
    time.sleep(0.3)

    # Layer 2: PX XML backup
    for col, vals in fetch_px_xml(pxd).items():
        pxd_add_list(col, vals)

    # Layer 3: Regex from paper
    if pub_dict:
        for col, vals in regex_extract(pub_dict).items():
            pxd_add_list(col, vals)

    # Layer 4: Global majority fallback for high-signal columns
    found_base = set(re.sub(r'\.\d+$', '', c) for c in pxd_vals.keys())
    for col in list(all_base - found_base):
        if non_na_ratio.get(col, 0.0) > 0.80:
            pxd_add(col, global_modes.get(col))

    # Handle Modification slots (.1, .2 etc)
    mods = list(dict.fromkeys(pxd_vals.pop('Characteristics[Modification]', [])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i == 0 else f'Characteristics[Modification].{i}'
        pxd_vals[slot] = [mod]

    # ── Per-file assignment ───────────────────────────────────────────────
    for i, (row_idx, raw_file) in enumerate(zip(idx, raw_files)):

        # Parse filename for per-file metadata
        fraction   = parse_fraction(raw_file)
        biol_rep   = parse_biol_replicate(raw_file)
        tech_rep   = parse_tech_replicate(raw_file)
        fn_label   = normalize_label_from_filename(raw_file)
        fn_conds   = parse_condition_from_filename(raw_file)

        for col in target_cols:
            base_col = re.sub(r'\.\d+$', '', col)

            # Per-file columns override PXD-level
            if col == 'Comment[FractionIdentifier]' and fraction:
                final_sub.at[row_idx, col] = fraction
                continue
            if col == 'Characteristics[BiologicalReplicate]' and biol_rep:
                final_sub.at[row_idx, col] = biol_rep
                continue
            if col == 'Characteristics[TechnicalReplicate]' and tech_rep:
                final_sub.at[row_idx, col] = tech_rep
                continue

            # Filename-derived label overrides PXD-level
            if col == 'Characteristics[Label]':
                if fn_label:
                    final_sub.at[row_idx, col] = fn_label
                elif pxd_vals.get(col):
                    final_sub.at[row_idx, col] = pxd_vals[col][0]
                continue

            # Filename condition overrides PXD-level for key columns
            if col in fn_conds:
                final_sub.at[row_idx, col] = fn_conds[col]
                continue

            # PXD-level value for everything else
            vals = pxd_vals.get(col) or pxd_vals.get(base_col) or []
            vals = [v for v in vals if str(v).strip().lower() not in ('not applicable','')]

            if vals:
                # Cycle values if multiple (handles multiple biological conditions)
                final_sub.at[row_idx, col] = vals[i % len(vals)]
            # else: remains 'Not Applicable'

    n_filled = sum((final_sub.loc[idx, col] != 'Not Applicable').sum()
                   for col in target_cols[:5])
    print(f'  {pxd}: {len(raw_files)} files, fraction_parsed={sum(1 for rf in raw_files if parse_fraction(rf))}')

# Final cleanup
final_sub = final_sub.fillna('Not Applicable')
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['TextSpan','nan','None','[]','','null','Not applicable','not available']
    )
    final_sub.loc[mask, col] = 'Not Applicable'

final_sub.to_csv(OUT_PATH, index=False)
print(f'\nSaved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')

Extracting PXDs:   7%|▋         | 1/15 [00:04<00:58,  4.21s/it]

  PXD004010: 10 files, fraction_parsed=0


Extracting PXDs:  13%|█▎        | 2/15 [00:06<00:37,  2.88s/it]

  PXD016436: 18 files, fraction_parsed=0


Extracting PXDs:  20%|██        | 3/15 [00:08<00:30,  2.52s/it]

...

  PXD062469: 32 files, fraction_parsed=0


Extracting PXDs:  93%|█████████▎| 14/15 [00:35<00:02,  2.75s/it]

  PXD062877: 48 files, fraction_parsed=0


Extracting PXDs: 100%|██████████| 15/15 [00:39<00:00,  2.61s/it]

  PXD064564: 30 files, fraction_parsed=0

Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_v2_per_file.csv
Shape : (1659, 81)


In [10]:
label_cols = [c for c in final_sub.columns
              if c not in ('ID','PXD','Raw Data File','Usage')]

rows = [(col, (final_sub[col] != 'Not Applicable').sum()) for col in label_cols]
rows.sort(key=lambda x: -x[1])

print(f'{"Column":<55} {"Filled":>7} {"Pct":>6}')
print('-' * 72)
for col, n in rows:
    if n > 0:
        print(f'{col:<55} {n:>7} {n/len(final_sub)*100:>5.1f}%')

zero = sum(1 for _, n in rows if n == 0)
print(f'\nFilled: {len(rows)-zero} / {len(rows)} columns')

print(f'\nPer-file parsing results:')
for col in ['Comment[FractionIdentifier]','Characteristics[BiologicalReplicate]',
            'Characteristics[Label]']:
    if col in final_sub.columns:
        n = (final_sub[col] != 'Not Applicable').sum()
        print(f'  {col:<50} {n:>5} / {len(final_sub)}')

print(f'\nSubmit {OUT_PATH.name} to Kaggle.')


Column                                                   Filled    Pct
------------------------------------------------------------------------
Characteristics[Bait]                                      1659 100.0%
Characteristics[BiologicalReplicate]                       1659 100.0%
Characteristics[CellPart]                                  1659 100.0%
Characteristics[CleavageAgent]                             1659 100.0%
Characteristics[Compound]                                  1659 100.0%
Characteristics[Depletion]                                 1659 100.0%
Characteristics[Label]                                     1659 100.0%
Characteristics[MaterialType]                              1659 100.0%
Characteristics[Modification]                              1659 100.0%
Characteristics[Modification].1                            1659 100.0%
Characteristics[Modification].2                            1659 100.0%
Characteristics[Modification].3                            1659 100.0%
Char

In [12]:
candidate_paths = []

# Prefer path produced by the pipeline cell
if 'OUT_PATH' in globals():
	candidate_paths.append(Path(OUT_PATH))

# Local and common fallback locations
candidate_paths.extend([
	Path('outputs/submission_v2_per_file.csv'),
	Path('../outputs/submission_v2_per_file.csv'),
	Path('/kaggle/working/submission_v2.csv'),
])

submission_path = next((p for p in candidate_paths if p.exists()), None)
if submission_path is None:
	raise FileNotFoundError(
		"submission file not found. Tried:\n" +
		"\n".join(str(p) for p in candidate_paths)
	)

df = pd.read_csv(submission_path)
print(f'Loaded: {submission_path}')
print(df['Comment[Instrument]'].value_counts().head(5))
print(df['Characteristics[BiologicalReplicate]'].value_counts().head(5))
print(df['Comment[FractionIdentifier]'].value_counts().head(10))

Loaded: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_v2_per_file.csv
Comment[Instrument]
AC=MS:1003094;NT=Orbitrap Exploris 480    1391
AC=MS:1003378;NT=Orbitrap Astral            63
AC=MS:1001911;NT=Q Exactive                 54
AC=MS:1002523;NT=Q Exactive HF              33
AC=MS:1000932;NT=TripleTOF 5600             32
Name: count, dtype: int64
Characteristics[BiologicalReplicate]
1    1619
2      32
3       8
Name: count, dtype: int64
Comment[FractionIdentifier]
1    1659
Name: count, dtype: int64


In [14]:
sample_path = SAMPLE_SUB if 'SAMPLE_SUB' in globals() else Path('data/SampleSubmission.csv')
sample = pd.read_csv(sample_path)

raw_files = sample['Raw Data File'].tolist()[:20]
for rf in raw_files:
    print(f"{rf:40s} → fraction={parse_fraction(rf)}")

ad_pl01.raw                              → fraction=None
ad_pl02.raw                              → fraction=None
ad_pl03.raw                              → fraction=None
ad_pl04.raw                              → fraction=None
ad_pl05.raw                              → fraction=None
ad_pl06.raw                              → fraction=None
ad_pl07.raw                              → fraction=None
ad_pl08.raw                              → fraction=None
ad_pl09.raw                              → fraction=None
ad_pl10.raw                              → fraction=None
pTn5053_1.raw                            → fraction=None
pTn5053_2.raw                            → fraction=None
pTn5053_3.raw                            → fraction=None
pTn5053_delta_Chi_1.raw                  → fraction=None
pTn5053_delta_Chi_2.raw                  → fraction=None
pTn5053_delta_Chi_3.raw                  → fraction=None
pTn5053_delta_ClpX_1.raw                 → fraction=None
pTn5053_delta_ClpX_2.raw       

In [17]:
df = pd.read_csv(OUT_PATH)

# FractionIdentifier: if all files in a PXD got the same value "1" 
# and parse_fraction found nothing, it's a fallback artifact — set to Not Applicable
# Check: if every row in a PXD has the same fraction value, it wasn't per-file parsed
for pxd, grp in df.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs) == 1 and fracs[0] == '1':
        # All same value = fallback artifact, not real fractionation
        df.loc[grp.index, 'Comment[FractionIdentifier]'] = 'Not Applicable'

n = (df['Comment[FractionIdentifier]'] != 'Not Applicable').sum()
print(f'FractionIdentifier filled after fix: {n} / {len(df)}')

out_file = Path('outputs/submission_v2_fixed.csv')
out_file.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_file, index=False)
print(f'Saved {out_file}')

FractionIdentifier filled after fix: 1659 / 1659
Saved outputs\submission_v2_fixed.csv


In [19]:
f = pd.read_csv('outputs/submission_v2_fixed.csv')

print("FractionIdentifier distribution:")
print(df['Comment[FractionIdentifier]'].value_counts().head(10))

print("\nPer-PXD unique fraction values:")
for pxd, grp in df.groupby('PXD'):
    fracs = sorted(grp['Comment[FractionIdentifier]'].unique().tolist())
    print(f"  {pxd} ({len(grp)} files): {fracs[:8]}")

FractionIdentifier distribution:
Comment[FractionIdentifier]
1    1659
Name: count, dtype: int64

Per-PXD unique fraction values:
  PXD004010 (10 files): [1]
  PXD016436 (18 files): [1]
  PXD019519 (6 files): [1]
  PXD025663 (12 files): [1]
  PXD040582 (24 files): [1]
  PXD050621 (9 files): [1]
  PXD061009 (2 files): [1]
  PXD061090 (6 files): [1]
  PXD061136 (2 files): [1]
  PXD061195 (1376 files): [1]
  PXD061285 (60 files): [1]
  PXD062014 (24 files): [1]
  PXD062469 (32 files): [1]
  PXD062877 (48 files): [1]
  PXD064564 (30 files): [1]


In [20]:
df = pd.read_csv('outputs/submission_v2_fixed.csv', dtype=str)

for pxd, grp in df.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs) == 1 and str(fracs[0]).strip() in ('1', '1.0'):
        df.loc[grp.index, 'Comment[FractionIdentifier]'] = 'Not Applicable'

n = (df['Comment[FractionIdentifier]'] != 'Not Applicable').sum()
print(f'FractionIdentifier filled after fix: {n} / {len(df)}')

df.to_csv('outputs/submission_v2_fixed.csv', index=False)
print('Saved')

FractionIdentifier filled after fix: 0 / 1659
Saved
